In [1]:
!pip install aerospike

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.1 MB/s  0:00:00m 13.1 MB/s eta 0:00:01


In [4]:
# docker network create aerospike-net
#
# docker run -d --name aerospike-vault \
#  --network aerospike-net \
#  -p 3000:3000 -p 3001:3001 -p 3002:3002 \
#  aerospike/aerospike-server

import aerospike
import sys

# 1. Connect to the 'Frontier' Vault
config = {'hosts': [('127.0.0.1', 3000)]}
try:
    client = aerospike.client(config).connect()
    print("Connection to Aerospike established.")
except Exception as e:
    print(f"Failed to connect: {e}")
    sys.exit(1)

# 2. Define the Data (The "Hot Cache")
# Structure: (Namespace, Set, User_Key)
players_to_cache = [
    {
        "key": ("test", "live_stats", "celebrini_71"),
        "bins": {
            "name": "Macklin Celebrini",
            "team": "San Jose Sharks",
            "goals": 1,
            "toi_sec": 1240,
            "status": "On Ice"
        }
    },
    {
        "key": ("test", "live_stats", "hughes_43"),
        "bins": {
            "name": "Quinn Hughes",
            "team": "Vancouver Canucks",
            "goals": 0,
            "toi_sec": 1520,
            "status": "Bench"
        }
    }
]

# 3. Write the Records (Sub-millisecond latency)
for player in players_to_cache:
    client.put(player["key"], player["bins"])
    print(f"Vaulted real-time data for: {player['bins']['name']}")

# 4. Perform a "Gouldian" Inquiry
# Let's retrieve Quinn Hughes specifically to check his Time on Ice
print("\n--- Live Inquiry Result ---")
(key, meta, record) = client.get(("test", "live_stats", "hughes_43"))

print(f"Player: {record['name']}")
print(f"Current Team: {record['team']}")
print(f"Live TOI: {record['toi_sec'] // 60} minutes")
print(f"Current Status: {record['status']}")

client.close()

Connection to Aerospike established.
Vaulted real-time data for: Macklin Celebrini
Vaulted real-time data for: Quinn Hughes

--- Live Inquiry Result ---
Player: Quinn Hughes
Current Team: Vancouver Canucks
Live TOI: 25 minutes
Current Status: Bench
